# 🤖 AI Engineering Fundamentals — Lezione 5
## Notebook Gruppo C

**ITS Novitas 4.0 | Giovedì 04/06/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "C"
MEMBRI = ["", "", "", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [ ]:
# Setup — eseguite questa cella per prima
# La API key viene letta dal file .env nella root del progetto (non più dai Secrets di Colab)
import anthropic, os, json, requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # carica ANTHROPIC_API_KEY dal .env

client = anthropic.Anthropic()          # legge automaticamente ANTHROPIC_API_KEY dall'ambiente
print("✅ Setup completato!")

---
## 🎯 Tema del Gruppo C: Tool Use & Loop Agente

Esplorate come il modello decide quale tool chiamare
e come gestire il loop agente — incluso il caso multi-step
dove il modello chiama più tool in sequenza.

---
### Esercizio 1 — Vedere la decisione del modello *(guidato)*

Prima di implementare il loop completo, osservate cosa restituisce
il modello quando decide di usare un tool.
La risposta NON è testo — è JSON con la decisione.

In [ ]:
# Esercizio 1 — osservare la decisione del modello

tool_calcolatrice = {
    "name": "calcola",
    "description": "Esegui un'operazione matematica precisa. Usa questo tool per qualsiasi calcolo aritmetico.",
    "input_schema": {
        "type": "object",
        "properties": {
            "espressione": {
                "type": "string",
                "description": "L'espressione matematica da calcolare. Es: '234 * 567'"
            }
        },
        "required": ["espressione"]
    }
}

# Chiamata CON tool — NON implementiamo il loop, solo osserviamo
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=500,
    tools=[tool_calcolatrice],
    messages=[{"role": "user", "content": "Quanto fa 1234 * 5678?"}]
)

print(f"stop_reason: {response.stop_reason}")
print(f"\nContenuto della risposta:")
for block in response.content:
    print(f"  tipo: {block.type}")
    if block.type == "tool_use":
        print(f"  nome tool: {block.name}")
        print(f"  parametri: {block.input}")
        print(f"  id: {block.id}")

print()
print("💡 Il modello NON ha calcolato nulla.")
print("   Ha solo detto: chiama 'calcola' con questa espressione.")
print("   Il calcolo lo fa Python — non Claude.")

---
### Esercizio 2 — Il loop completo *(guidato)*

Implementate il loop agente completo con il contatore
di sicurezza per evitare loop infiniti.

In [ ]:
# Esercizio 2 — tool loop completo

def calcola(espressione: str) -> str:
    """Calcolatrice sicura con lista bianca di caratteri."""
    allowed = set('0123456789+-*/().% ')
    if not all(c in allowed for c in espressione):
        return "Errore: espressione non valida"
    try:
        return f"{espressione} = {eval(espressione)}"
    except Exception as e:
        return f"Errore: {str(e)}"

def esegui_tool(nome, parametri):
    """Router: smista la chiamata al tool giusto."""
    if nome == "calcola":
        return calcola(parametri["espressione"])
    return f"Tool '{nome}' non trovato"

MAX_ITERAZIONI = 10

def chat_con_tool(messaggio, tools):
    """Chatbot con tool loop e contatore di sicurezza."""
    history = [{"role": "user", "content": messaggio}]
    iterazioni = 0

    while True:
        iterazioni += 1
        # Contatore di sicurezza: evita loop infiniti
        if iterazioni > MAX_ITERAZIONI:
            return "Errore: loop agente non terminato"

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=tools,
            messages=history
        )

        # Il modello ha finito: restituiamo il testo della risposta
        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text")

        # Il modello vuole usare uno o più tool: li eseguiamo e rimandiamo i risultati
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_tool(block.name, block.input)
                    print(f"  ✅ {risultato}")
                    # Ogni risultato deve riferirsi al tool_use tramite il suo id
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": risultato,
                    })
            history.append({"role": "user", "content": tool_results})

# Test
print("❓ Quanto fa 1234 * 5678?")
print(chat_con_tool("Quanto fa 1234 * 5678?", [tool_calcolatrice]))

---
### Esercizio 3 — Multi-step: tool concatenati *(libero)*

Fate una domanda che richiede 2 tool in sequenza.
Osservate quante iterazioni fa il loop e in che ordine
vengono chiamati i tool.

In [ ]:
# Esercizio 3 — multi-step con tool concatenati

# Tool Wikipedia per cercare informazioni
tool_wikipedia = {
    "name": "cerca_wikipedia",
    "description": "Cerca informazioni su Wikipedia. Usa per fatti, biografie, concetti tecnici.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Termine da cercare"}
        },
        "required": ["query"]
    }
}

def cerca_wikipedia(query: str) -> str:
    try:
        url = f"https://it.wikipedia.org/api/rest_v1/page/summary/{query.replace(' ', '_')}"
        r = requests.get(url, timeout=5)
        if r.status_code == 200:
            return r.json().get("extract", "")[:500]
        return f"Nessun risultato per '{query}'"
    except Exception as e:
        return f"Errore: {str(e)}"

# Aggiornate il router
def esegui_tool_v2(nome, parametri):
    if nome == "calcola":          return calcola(parametri["espressione"])
    if nome == "cerca_wikipedia":  return cerca_wikipedia(parametri["query"])
    return f"Tool '{nome}' non trovato"

# Versione del loop che usa esegui_tool_v2 e conta le iterazioni
def chat_con_tool_v2(messaggio, tools):
    history = [{"role": "user", "content": messaggio}]
    iterazioni = 0
    while True:
        iterazioni += 1
        if iterazioni > MAX_ITERAZIONI:
            return "Errore: loop agente non terminato", iterazioni
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=tools,
            messages=history
        )
        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text"), iterazioni
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_tool_v2(block.name, block.input)
                    print(f"  ✅ {str(risultato)[:100]}")
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": risultato})
            history.append({"role": "user", "content": tool_results})

domande_multistep = [
    # Richiede Wikipedia + calcolatrice
    "Anthropic è stata fondata nel 2021. Quanti anni ha adesso (siamo nel 2026)?",
    # Richiede 2 ricerche Wikipedia
    "Chi ha fondato Anthropic e chi ha fondato OpenAI?",
    # La vostra domanda multi-step
    "Quanti anni separano la fondazione di OpenAI (2015) da quella di Anthropic (2021)?",
]

TUTTI_I_TOOL = [tool_calcolatrice, tool_wikipedia]

for domanda in domande_multistep:
    if domanda:
        print(f"\n{'='*55}")
        print(f"❓ {domanda}")
        risposta, n_iter = chat_con_tool_v2(domanda, TUTTI_I_TOOL)
        print(f"🤖 {risposta}")
        print(f"   (iterazioni del loop: {n_iter})")

# Conclusione: le domande multi-step richiedono più iterazioni (una per ogni "round"
# di tool). Il modello chiama prima i tool per raccogliere i dati (es. ricerca, oppure
# i due anni) e solo dopo, con i risultati in mano, esegue il calcolo o sintetizza la
# risposta finale. L'ordine segue le dipendenze: prima i dati, poi l'elaborazione.

---
### Esercizio 4 — La description conta *(libero)*

Cambiate la description del tool calcolatrice in modo vago.
Il modello lo usa ancora correttamente?
Dimostrate che la description è la parte più critica della definizione.

In [ ]:
# Esercizio 4 — l'impatto della description

tool_vago = {
    "name": "calcola",
    "description": "Tool.",  # ← description completamente vaga
    "input_schema": tool_calcolatrice["input_schema"]
}

tool_fuorviante = {
    "name": "calcola",
    "description": "Usa questo tool per rispondere a domande generali.",  # ← fuorviante
    "input_schema": tool_calcolatrice["input_schema"]
}

domande_test = [
    "Quanto fa 15% di 847?",
    "Qual è la radice quadrata di 144?",
    "Fammi una poesia sulla Sardegna.",  # non dovrebbe usare la calcolatrice
]

def usa_il_tool(domanda, tool_def):
    """Ritorna True se il modello decide di chiamare il tool per questa domanda."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        tools=[tool_def],
        messages=[{"role": "user", "content": domanda}],
    )
    return any(b.type == "tool_use" for b in response.content)

for nome, td in [("PRECISA", tool_calcolatrice), ("VAGA", tool_vago), ("FUORVIANTE", tool_fuorviante)]:
    print(f"\n{'='*55}")
    print(f"Description {nome}: \"{td['description']}\"")
    for d in domande_test:
        usato = usa_il_tool(d, td)
        print(f"  {'🔧 usa calcola' if usato else '💬 risponde diretto'} ← {d}")

# Conclusione:
# La description impatta MOLTO il comportamento. Con una description precisa il modello
# usa la calcolatrice per i calcoli e NON la usa per la poesia. Con una description vaga
# ("Tool.") o fuorviante ("per domande generali") il modello sceglie peggio: può non
# usarla quando servirebbe o usarla a sproposito (es. per la poesia).
# Regola pratica: la description deve dire CHIARAMENTE cosa fa il tool e QUANDO usarlo
# (e quando NON usarlo) — è la parte più importante della definizione di un tool.

---
## 📊 Preparate la presentazione (5 slide)

1. **Il modello decide, Python esegue** — mostrate il JSON della decisione
2. **Il tool loop** — le 3 fasi: aggiungi assistant, esegui tool, aggiungi risultati
3. **Multi-step** — quante iterazioni per ogni domanda?
4. **La description conta** — risultati del confronto vago/preciso/fuorviante
5. **La vostra regola pratica** — come scrivere una buona description

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*